# Fase 7 — Varian **P1-NoProj** (penutup celah metodologis)

Menutup serangan reviewer paling mungkin: *"DefMamba dapat +1.7% **dengan** menambah 13,7%
parameter; Anda memotong `ssm_ratio` 2.0→1.4, jadi wajar gagal — Anda menghapus sumber gain-nya."*

**Dua perubahan, keduanya MENGUNTUNGKAN deformable:**
1. `ssm_ratio` dikembalikan ke **2.0** (sama B2 — SSM tidak lagi dihandicap 30%)
2. `ConvOffset.proj` (Conv2d C→C 1×1) **dibuang** — layer ini **tidak ada di DefMamba**, dan
   menyumbang 57–64% biaya offset network (skalanya C²)

| Varian | Params | Δ vs B2 | ssm_ratio |
|---|---|---|---|
| B2 (baseline) | 5.986.912 | — | 2.0 |
| P1 (lama, capped) | 5.930.869 | −0.94% | 1.4 |
| **P1-NoProj (ini)** | **6.329.230** | **+5.72%** | **2.0** |
| *(P1-Uncapped — DITOLAK)* | *7.220.686* | *+20.61%* | *2.0* |

> ⚠️ **+5.72% MELEBIHI toleransi ±5% — DISENGAJA.** Framing paper: *stress test yang condong
> menguntungkan deformable*, BUKAN perbandingan setara. Nyatakan eksplisit, jangan disamarkan.

> ⚠️ **P1-NoProj adalah arsitektur BERBEDA dari P1**, bukan "P1 tanpa capping". Laporkan sebagai
> varian tersendiri — **jangan gabungkan seed-nya dengan hasil P1 yang sudah ada.**

## Cara membaca hasil — TENTUKAN SEBELUM TRAINING (hindari post-hoc)

| Hasil | Interpretasi | Nilai |
|---|---|---|
| P1-NoProj **tetap kalah** dari B2 | Mekanisme gagal **bahkan dgn +5,7% param & SSM penuh** | 🟢🟢 Terkuat — serangan tertutup rapat |
| P1-NoProj **≈ B2** (tak signifikan) | Sama dgn temuan sekarang, di kondisi lebih menguntungkan | 🟢 Memperkuat null |
| P1-NoProj **menang** atas B2 | Gain muncul hanya saat param dibiarkan tumbuh | 🟡 Tetap temuan bagus — bukti gain DefMamba dari KAPASITAS, bukan mekanisme. Wajib dipasangkan dgn FPS. |

**Ketiganya bermanfaat — tidak ada hasil yang "merusak" paper.**

## Anggaran

Seed **1–3 dulu**, lalu nilai. Ingat pelajaran proyek: **seed1 ekstrem SELALU regresi ke mean**
(teramati 4×) — jangan simpulkan apa pun sebelum minimal 3 seed.

## Progres

| # | Model | Seed | Status |
|---|---|---|---|
| 1 | P1-NoProj | 1 | ⬜ |
| 2 | P1-NoProj | 2 | ⬜ |
| 3 | P1-NoProj | 3 | ⬜ |
| 4 | P1-NoProj | 4 | ⬜ (opsional) |
| 5 | P1-NoProj | 5 | ⬜ (opsional) |


## 1. Mount Drive + Kredensial Kaggle (Cache)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, stat, shutil

DRIVE_ROOT = "/content/drive/MyDrive/demamba-yolowd"
os.makedirs(DRIVE_ROOT, exist_ok=True)
KAGGLE_JSON_CACHE = f"{DRIVE_ROOT}/kaggle.json"

os.makedirs("/root/.kaggle", exist_ok=True)
assert os.path.exists(KAGGLE_JSON_CACHE), (
    f"kaggle.json tidak ada di {KAGGLE_JSON_CACHE} -- jalankan fase-setup-colab-mamba.ipynb dulu."
)
shutil.copy(KAGGLE_JSON_CACHE, "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", stat.S_IRUSR | stat.S_IWUSR)
print("kaggle.json dimuat dari cache Drive.")

Mounted at /content/drive
kaggle.json dimuat dari cache Drive.


## 2. Tarik Dataset dari Kaggle + Deteksi GPU

In [ ]:
import subprocess, sys, glob, torch

# Kontrol numerik lintas-GPU: cudnn.allow_tf32 default TRUE di torch, tapi TF32 cuma ada
# di hardware Ampere+ (A100, L4) -- TIDAK ADA di Turing (T4). Artinya B2/P1 seed0 (Kaggle T4)
# dan seed1+ (Colab, GPU bisa A100/L4) TIDAK menjalankan presisi numerik yang identik kalau
# dibiarkan default -- variabel yang belum terkontrol. Dimatikan di sini demi konsistensi
# lintas-platform, ditemukan setelah B2 seed1 (A100) turun di 7/8 kelas vs seed0 (T4).
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
print(f"TF32 dimatikan (matmul={torch.backends.cuda.matmul.allow_tf32}, cudnn={torch.backends.cudnn.allow_tf32})")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)
os.makedirs("/content/data", exist_ok=True)
os.makedirs("/content/deform_src", exist_ok=True)

subprocess.run(["kaggle", "datasets", "download", "-d", "benyxu/dsf-yolo-steel-tube-dataset",
                 "-p", "/content/data", "--unzip"], check=True)
subprocess.run(["kaggle", "datasets", "download", "-d", "benyxu/deform-ss2d-py",
                 "-p", "/content/deform_src", "--unzip"], check=True)

DEFORM_SRC = glob.glob("/content/deform_src/**/deform_ss2d.py", recursive=True)[0]
DSF_BASE = "/content/data/steel-tube-dataset-all/yolo"
assert os.path.exists(f"{DSF_BASE}/images/train2021")
print(f"DEFORM_SRC = {DEFORM_SRC}\nDSF_BASE   = {DSF_BASE}")

print(f"\nGPU: {torch.cuda.get_device_name(0)}  (compute cap {torch.cuda.get_device_capability(0)})")
print("(GPU cepat -> mungkin selesai 1 sesi. GPU lambat/T4 -> auto-resume kalau kepotong, lihat Sel 7)")

TF32 dimatikan (matmul=False, cudnn=False)


## 3. 🔧 KONFIGURASI — Edit Ini Setiap Kali Ganti Kombinasi

In [ ]:
# ============================================================
#  UBAH BARIS INI SEBELUM "Run all"
# ============================================================
MODEL = "P1-NoProj"   # fase ini fokus varian baru; "B2"/"P1" tetap bisa dipakai utk regresi
SEED  = 5             # 1-5. Saran: 1-3 dulu, nilai, baru lanjut 4-5 bila perlu.
LR0 = 0.002   # WAJIB diturunkan dari default 0.01 -- lr0=0.01 terbukti tidak stabil utk P1+STE.
              # Bukti: linear rusak @epoch82 (LR saat itu ~0.0046), kosinus rusak LEBIH CEPAT
              # @ epoch7 (LR saat itu ~0.00995, msh dekat lr0 penuh -- kosinus menahan LR tinggi
              # lebih lama di awal, bukan menurunkannya lebih cepat spt diduga sebelumnya).
              # Ambang bahaya empiris ~0.0046-0.0054. lr0=0.002 dimulai DI BAWAH ambang itu
              # sejak epoch 1, bukan menurun ke arahnya -- margin aman, bukan mepet.
              # Dipakai jg utk BackboneOnly/NeckOnly (sama-sama punya blok DeformSS2D dgn STE,
              # belum ada bukti ambang beda -- pakai yg sudah terbukti aman utk P1 dulu).
USE_COS_LR = False   # KEMBALI ke linear -- kosinus TERBUKTI LEBIH BURUK (rusak epoch 7 vs 70+),
                     # krn kurvanya datar di awal (menahan LR dekat lr0 lebih lama), padahal yang
                     # dibutuhkan adalah LR rendah SEJAK AWAL, bukan cuma di akhir. Linear + LR0
                     # rendah menyerang akar masalah (nilai LR), bukan cuma bentuk kurvanya.
# ============================================================

TARGET_EPOCHS = 150

# Peta konfigurasi per varian -- YAML, target params (hasil verifikasi capacity-matching
# offline, lihat tracker), jumlah DeformSS2D yg diharapkan (utk pre-flight Sel 5).
MODEL_CONFIG = {
    "B2":           {"yaml": "Mamba-YOLO-T.yaml",                    "params": 5.987, "ndef": 0},
    "P1":           {"yaml": "DeMamba-YOLO-T.yaml",                  "params": 5.931, "ndef": 10},
    "BackboneOnly": {"yaml": "DeMamba-YOLO-T-backbone-only.yaml",    "params": 5.952, "ndef": 6},
    "NeckOnly":     {"yaml": "DeMamba-YOLO-T-neck-only.yaml",        "params": 5.966, "ndef": 4},
    # Ablasi B-lanjut: stage-wise (lihat literatur-deformable-lintas-paradigma.md SS6.3).
    # Neck dibiarkan B2 di semua varian ini -- pertanyaan yg diuji adalah "stage backbone
    # mana", bukan neck lagi (isolasi bersih dari pertanyaan Ablasi B sebelumnya).
    "BB-Deep":      {"yaml": "DeMamba-YOLO-T-bb-deep.yaml",          "params": 5.991, "ndef": 1},
    "BB-Late":      {"yaml": "DeMamba-YOLO-T-bb-late.yaml",          "params": 5.964, "ndef": 4},
    "BB-Early":     {"yaml": "DeMamba-YOLO-T-bb-early.yaml",         "params": 5.975, "ndef": 2},
    # BERSYARAT (backlog) -- lihat tracker Section 2d. Sumbu KETIGA: kerapatan intra-stage.
    # Posisi 5 = pola D-S-D (2/3 blok deform, 1 tetap biasa), posisi 7 = deform penuh.
    # Neck TIDAK disentuh (sama seperti semua varian BB-* lainnya).
    "BB-Late-Alt":  {"yaml": "DeMamba-YOLO-T-bb-late-alt.yaml",      "params": 5.973, "ndef": 3},
    # Fase 7 -- varian BARU. Params SENGAJA +5.72% vs B2 (di luar toleransi +/-5%);
    # ssm_ratio 2.0 (tidak dihandicap) + ConvOffset.proj dibuang (setia ke DefMamba).
    "P1-NoProj":    {"yaml": "DeMamba-YOLO-T-noproj.yaml",           "params": 6.329, "ndef": 10},
}
assert MODEL in MODEL_CONFIG, f"MODEL harus salah satu dari {list(MODEL_CONFIG)}, dapat: {MODEL}"
assert SEED in (1, 2, 3, 4, 5), f"SEED harus 1-5, dapat: {SEED}"

RUN_NAME = f"{MODEL.lower()}_seed{SEED}_150ep"
if USE_COS_LR:
    RUN_NAME += "_coslr"
if LR0 != 0.01:
    RUN_NAME += f"_lr{LR0}"
# nama run beda kalau konfigurasi beda -- supaya TIDAK ke-resume checkpoint dgn train_args lama
# yang tidak sesuai (mis. lr0/cos_lr beda), harus mulai training dari nol tiap konfigurasi baru
CONFIG_YAML = MODEL_CONFIG[MODEL]["yaml"]
EXPECTED_PARAMS = MODEL_CONFIG[MODEL]["params"]
EXPECTED_NDEF = MODEL_CONFIG[MODEL]["ndef"]
DRIVE_OUTPUT = f"{DRIVE_ROOT}/output"
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

print(f"MODEL={MODEL}  SEED={SEED}  RUN_NAME={RUN_NAME}  target={TARGET_EPOCHS} epoch  lr0={LR0}  cos_lr={USE_COS_LR}")


## 4. Clone Mamba-YOLO + Semua Patch + Build Kernel

Identik Fase 2A-Colab/2B-Colab -- 8 patch registrasi + 2 patch `weights_only` (`tasks.py`) +
1 patch `weights_only` (`torch_utils.py`), kernel vendor-only.

In [ ]:
import re, py_compile

REPO = "/content/Mamba-YOLO"
# WAJIB sebelum rmtree: pastikan cwd proses TIDAK sedang di dalam REPO (bisa terjadi kalau
# notebook di-run ulang di kernel yang sama setelah sel pre-flight sempat os.chdir(REPO) --
# menghapus folder yang sedang "ditempati" proses bikin SEMUA subprocess berikutnya gagal
# dgn "fatal: Unable to read current working directory").
os.chdir("/content")
if os.path.exists(REPO):
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--depth", "1", "https://github.com/HZAI-ZJNU/Mamba-YOLO.git", REPO], check=True)

DEFORM = f"{REPO}/ultralytics/nn/modules/deform_ss2d.py"
shutil.copy(DEFORM_SRC, DEFORM)

src = open(DEFORM).read()
checks = {
    "STE (gradien Delta-p)": "x_soft" in src,
    "ssm_ratio=1.4 (capacity match P1)": "ssm_ratio=1.4" in src,
    "subclass block terdaftar": "class DeformVSSBlock" in src and "class DeformXSSBlock" in src,
}
if not all(checks.values()):
    raise RuntimeError("deform_ss2d.py BUKAN versi final -- cek dataset 'deform-ss2d-py' di Kaggle.")
print("Tripwire deform_ss2d.py OK")

def patch(path, subs):
    p = f"{REPO}/{path}"
    src = open(p).read()
    for old, new, tag in subs:
        if new in src:
            print(f"  ~ {tag}: skip"); continue
        assert src.count(old) == 1, f"GAGAL {tag}: {src.count(old)}x (harus 1x)"
        src = src.replace(old, new)
        print(f"  OK {tag}")
    open(p, "w").write(src)

print("\n### Patch")
patch("ultralytics/nn/modules/mamba_yolo.py", [
    ("class XSSBlock(nn.Module):\n    def __init__(",
     "class XSSBlock(nn.Module):\n    SSM_CLASS = SS2D\n    def __init__(", "1.1"),
    ("nn.Sequential(*(SS2D(d_model=self.hidden_dim,",
     "nn.Sequential(*(self.SSM_CLASS(d_model=self.hidden_dim,", "1.2"),
    ("class VSSBlock(nn.Module):\n    def __init__(",
     "class VSSBlock(nn.Module):\n    SSM_CLASS = SS2D\n    def __init__(", "1.3"),
    ("            self.op = SS2D(",
     "            self.op = self.SSM_CLASS(", "1.4"),
])
patch("ultralytics/nn/modules/deform_ss2d.py", [
    ("from .mamba_yolo import SS2D",
     "from .mamba_yolo import SS2D, VSSBlock, XSSBlock", "2.1"),

    # ---- FASE 7: Patch P1-NoProj (PATCH-p1-noproj.md, Patch 1-3) ----
    # PRINSIP: semua argumen baru DEFAULT ke perilaku lama -> hasil 5-seed P1 /
    # BackboneOnly / NeckOnly / BB-* yang sudah terkumpul tetap reproducible
    # bit-per-bit. Dijamin oleh cek REGRESI di pre-flight (P1 lama = 5.930.869).
    ("    def __init__(self, in_ch: int, kernel_size: int):\n"
     "        super().__init__()\n"
     "        pad = kernel_size // 2 if kernel_size > 1 else 0\n"
     "        self.proj = nn.Conv2d(in_ch, in_ch, kernel_size=1)",
     "    def __init__(self, in_ch: int, kernel_size: int, use_proj: bool = True):\n"
     "        super().__init__()\n"
     "        pad = kernel_size // 2 if kernel_size > 1 else 0\n"
     "        # use_proj=False -> lebih setia ke DefMamba (offset net mereka tidak\n"
     "        # punya proyeksi C->C di depan, Fig.3c). Identity = 0 param.\n"
     "        self.use_proj = use_proj\n"
     "        self.proj = nn.Conv2d(in_ch, in_ch, kernel_size=1) if use_proj else nn.Identity()",
     "NP-1 ConvOffset.use_proj"),

    ("    def __init__(self, embed_dim: int, kernel_size: int = 5):\n"
     "        super().__init__()\n"
     "        self.conv_offset = ConvOffset(embed_dim, kernel_size)",
     "    def __init__(self, embed_dim: int, kernel_size: int = 5, use_proj: bool = True):\n"
     "        super().__init__()\n"
     "        self.conv_offset = ConvOffset(embed_dim, kernel_size, use_proj=use_proj)",
     "NP-2 DeformableScan.use_proj"),

    ("                 deform_kernel_size: int = None,\n"
     "                 **kwargs):",
     "                 deform_kernel_size: int = None,\n"
     "                 deform_use_proj: bool = True,\n"
     "                 **kwargs):",
     "NP-3a DeformSS2D arg"),

    ("        self.deform_scans = nn.ModuleList([\n"
     "            DeformableScan(d_inner, kernel_size=ks) for _ in range(n_deform)\n"
     "        ])",
     "        self.deform_use_proj = deform_use_proj\n"
     "        self.deform_scans = nn.ModuleList([\n"
     "            DeformableScan(d_inner, kernel_size=ks, use_proj=deform_use_proj)\n"
     "            for _ in range(n_deform)\n"
     "        ])",
     "NP-3b teruskan use_proj"),
])

# Patch 4: kelas wrapper BARU (TAMBAH di akhir file, JANGAN ganti yang lama).
# PENTING: DeformVSSBlock lama meng-override __init__ untuk MEMAKSA ssm_ratio=1.4.
# Kelas *NP SENGAJA TIDAK meng-override __init__ -> ssm_ratio mengalir apa adanya
# dari YAML/VSSBlock (2.0). Jangan salin pola override itu ke sini.
_np_lines = [
    "",
    "",
    "class DeformVSSBlockNP(VSSBlock):",
    "    # VSSBlock varian P1-NoProj (backbone).",
    "    #   - ssm_ratio TETAP 2.0 (sama B2) -- TIDAK override __init__",
    "    #   - ConvOffset.proj dibuang (use_proj=False) -- lebih setia ke DefMamba",
    "    # Total 6.329.230 params (+5,72% vs B2). Melebihi toleransi +/-5% SECARA",
    "    # SENGAJA: stress test yang condong menguntungkan deformable.",
    "    SSM_CLASS = partial(DeformSS2D, deform_mode=\'k3_replace\',",
    "                        deform_kernel_size=5, deform_use_proj=False)",
    "",
    "",
    "class DeformXSSBlockNP(XSSBlock):",
    "    # XSSBlock varian P1-NoProj (neck). Lihat DeformVSSBlockNP.",
    "    SSM_CLASS = partial(DeformSS2D, deform_mode=\'k3_replace\',",
    "                        deform_kernel_size=5, deform_use_proj=False)",
    "",
]
_dsp = f"{REPO}/ultralytics/nn/modules/deform_ss2d.py"
_src = open(_dsp).read()
if "class DeformVSSBlockNP" not in _src:
    open(_dsp, "w").write(_src + "\n".join(_np_lines))
    print("  OK NP-4 kelas DeformVSSBlockNP / DeformXSSBlockNP ditambahkan")
else:
    print("  ~ NP-4: skip (sudah ada)")

patch("ultralytics/nn/tasks.py", [
    ("    XSSBlock\n)\n",
     "    XSSBlock\n)\nfrom ultralytics.nn.modules.deform_ss2d import DeformVSSBlock, DeformXSSBlock, DeformVSSBlockNP, DeformXSSBlockNP\n", "3.1"),
    ("SimpleStem, VisionClueMerge, VSSBlock, XSSBlock\n        }:",
     "SimpleStem, VisionClueMerge, VSSBlock, XSSBlock, DeformVSSBlock, DeformXSSBlock, DeformVSSBlockNP, DeformXSSBlockNP\n        }:", "3.2"),
    ("RepC3, XSSBlock}:",
     "RepC3, XSSBlock, DeformXSSBlock, DeformXSSBlockNP}:", "3.3"),
    ('            ckpt = torch.load(file, map_location="cpu")\n\n    except ModuleNotFoundError as e:',
     '            ckpt = torch.load(file, map_location="cpu", weights_only=False)\n\n    except ModuleNotFoundError as e:', "4.1"),
    ('        check_requirements(e.name)  # install missing module\n        ckpt = torch.load(file, map_location="cpu")',
     '        check_requirements(e.name)  # install missing module\n        ckpt = torch.load(file, map_location="cpu", weights_only=False)', "4.2"),
])
patch("ultralytics/utils/torch_utils.py", [
    ('    x = torch.load(f, map_location=torch.device("cpu"))\n    if "model" not in x:',
     '    x = torch.load(f, map_location=torch.device("cpu"), weights_only=False)\n    if "model" not in x:', "5.1"),
])

y = open(f"{REPO}/ultralytics/cfg/models/mamba-yolo/Mamba-YOLO-T.yaml").read()
y = re.sub(r"\bVSSBlock\b", "DeformVSSBlock", y)
y = re.sub(r"\bXSSBlock\b", "DeformXSSBlock", y)
open(f"{REPO}/ultralytics/cfg/models/mamba-yolo/DeMamba-YOLO-T.yaml", "w").write(y)

# Ablasi B: 2 varian campuran -- backbone-saja-deformable, neck-saja-deformable.
# Split di penanda "# Mamba-YOLO PAFPN" (awal section head/neck), regex-swap CUMA
# di satu bagian. Params sudah diverifikasi offline: BackboneOnly=5.952M(-0.58%),
# NeckOnly=5.966M(-0.36%) vs B2 -- keduanya dlm toleransi +-5% TANPA ssm_ratio baru.
base_yaml = open(f"{REPO}/ultralytics/cfg/models/mamba-yolo/Mamba-YOLO-T.yaml").read()
split_marker = "# Mamba-YOLO PAFPN"
idx = base_yaml.find(split_marker)
assert idx != -1, "penanda split YAML tidak ketemu -- struktur Mamba-YOLO-T.yaml berubah?"
backbone_part, head_part = base_yaml[:idx], base_yaml[idx:]
assert "VSSBlock" in backbone_part and "XSSBlock" not in backbone_part
assert "XSSBlock" in head_part and "VSSBlock" not in head_part

backbone_deform = re.sub(r"\bVSSBlock\b", "DeformVSSBlock", backbone_part)
open(f"{REPO}/ultralytics/cfg/models/mamba-yolo/DeMamba-YOLO-T-backbone-only.yaml", "w").write(
    backbone_deform + head_part)

head_deform = re.sub(r"\bXSSBlock\b", "DeformXSSBlock", head_part)
open(f"{REPO}/ultralytics/cfg/models/mamba-yolo/DeMamba-YOLO-T-neck-only.yaml", "w").write(
    backbone_part + head_deform)
print("  OK 2 YAML Ablasi B (backbone-only, neck-only) berhasil digenerate")

# Ablasi B-lanjut: 3 varian stage-wise di backbone (literatur-deformable-lintas-paradigma.md
# SS6.3). Neck TIDAK disentuh (tetap XSSBlock biasa di semua 3 varian ini) -- isolasi bersih
# dari pertanyaan Ablasi B sebelumnya. Tiap baris backbone punya komentar posisi unik
# ("# 1","# 3","# 5","# 7") dipakai utk menarget baris tepat, assert-guarded (fail loud
# kalau match != 1x, mis. krn upstream ubah format YAML).
def _swap_positions(text, positions):
    lines = text.split("\n")
    hit = {p: 0 for p in positions}
    for i, line in enumerate(lines):
        stripped = line.rstrip()
        for p in positions:
            if stripped.endswith(f"# {p}") and "VSSBlock" in line and "Deform" not in line:
                lines[i] = line.replace("VSSBlock", "DeformVSSBlock")
                hit[p] += 1
    for p in positions:
        assert hit[p] == 1, f"posisi {p}: ditemukan {hit[p]}x (harus tepat 1x) -- format YAML berubah?"
    return "\n".join(lines)

stagewise_variants = {"bb-deep": [7], "bb-late": [5, 7], "bb-early": [1, 3]}
for vname, positions in stagewise_variants.items():
    result = _swap_positions(base_yaml, positions)
    open(f"{REPO}/ultralytics/cfg/models/mamba-yolo/DeMamba-YOLO-T-{vname}.yaml", "w").write(result)
print(f"  OK {len(stagewise_variants)} YAML Ablasi B-lanjut (bb-deep, bb-late, bb-early) berhasil digenerate")

# BB-Late-Alt (BERSYARAT, backlog Section 2d) -- sumbu kerapatan intra-stage. Posisi 5
# dipecah jadi 3 baris terpisah (D-S-D) krn parse_model bikin n INSTANCE TERPISAH dari
# kelas yg sama per baris (bukan loop internal) -- lihat tasks.py baris ~946. Memecah 1
# baris jadi 3 menggeser SEMUA indeks "from" eksplisit di bawahnya +2 -- WAJIB diperbaiki,
# assert-guarded (kalau salah, parse_model gagal keras karena channel Concat tak cocok,
# bukan diam-diam salah). Sudah diverifikasi offline: save-layers hasil parse_model =
# [3,7,10,13,16,19,22], cocok persis perhitungan manual.
_lines = base_yaml.split("\n")
_t5 = None
for _i, _l in enumerate(_lines):
    if _l.rstrip().endswith("# 5") and "VSSBlock" in _l and "Deform" not in _l:
        assert _t5 is None, "posisi 5: >1 baris cocok"
        _t5 = _i
assert _t5 is not None, "posisi 5 tidak ditemukan"
_dsd = [
    "  - [-1, 1, DeformVSSBlock, [512]]      # 5a (D)",
    "  - [-1, 1, VSSBlock, [512]]             # 5b (S)",
    "  - [-1, 1, DeformVSSBlock, [512]]      # 5c (D)",
]
_lines = _lines[:_t5] + _dsd + _lines[_t5+1:]
_hit7 = 0
for _i, _l in enumerate(_lines):
    if _l.rstrip().endswith("# 7") and "VSSBlock" in _l and "Deform" not in _l:
        _lines[_i] = _l.replace("VSSBlock", "DeformVSSBlock")
        _hit7 += 1
assert _hit7 == 1, f"posisi 7: {_hit7}x"
_text = "\n".join(_lines)
_reindex = [
    ("  - [[-1, 5], 1, Concat, [1]]  # cat backbone P4",
     "  - [[-1, 7], 1, Concat, [1]]  # cat backbone P4 (geser +2 krn split posisi 5)"),
    ("  - [[-1, 11], 1, Concat, [1]]  # cat head P4",
     "  - [[-1, 13], 1, Concat, [1]]  # cat head P4 (geser +2 krn split posisi 5)"),
    ("  - [[-1, 8], 1, Concat, [1]]  # cat head P5",
     "  - [[-1, 10], 1, Concat, [1]]  # cat head P5 (geser +2 krn split posisi 5)"),
    ("  - [[14, 17, 20], 1, Detect, [nc]]  # Detect(P3, P4, P5)",
     "  - [[16, 19, 22], 1, Detect, [nc]]  # Detect(P3, P4, P5) (geser +2 krn split posisi 5)"),
]
for _old, _new in _reindex:
    assert _text.count(_old) == 1, f"GAGAL cari saat reindex: {_old!r}"
    _text = _text.replace(_old, _new)
open(f"{REPO}/ultralytics/cfg/models/mamba-yolo/DeMamba-YOLO-T-bb-late-alt.yaml", "w").write(_text)
print("  OK 1 YAML BB-Late-Alt (D-S-D, bersyarat) berhasil digenerate, indeks head diperbaiki")

# --- FASE 7 (Patch 6): YAML P1-NoProj. Salin YAML P1 (semua posisi deformable:
# 1,3,5,7,11,14,17,20) lalu ganti nama kelas ke varian *NP. ssm_ratio TIDAK
# disentuh di YAML -- dibiarkan default 2.0 seperti B2 (itu inti varian ini). ---
_p1 = open(f"{REPO}/ultralytics/cfg/models/mamba-yolo/DeMamba-YOLO-T.yaml").read()
_np = _p1.replace("DeformVSSBlock", "DeformVSSBlockNP").replace("DeformXSSBlock", "DeformXSSBlockNP")
assert _np.count("DeformVSSBlockNP") == 4, f"DeformVSSBlockNP bukan 4: {_np.count('DeformVSSBlockNP')}"
assert _np.count("DeformXSSBlockNP") == 4, f"DeformXSSBlockNP bukan 4: {_np.count('DeformXSSBlockNP')}"
assert "ssm_ratio" not in _np, "YAML tidak boleh memuat ssm_ratio -- biarkan default 2.0"
open(f"{REPO}/ultralytics/cfg/models/mamba-yolo/DeMamba-YOLO-T-noproj.yaml", "w").write(_np)
print("  OK 1 YAML P1-NoProj digenerate (4 DeformVSSBlockNP + 4 DeformXSSBlockNP)")

for f in ["ultralytics/nn/modules/mamba_yolo.py", "ultralytics/nn/modules/deform_ss2d.py",
          "ultralytics/nn/tasks.py", "ultralytics/utils/torch_utils.py"]:
    py_compile.compile(f"{REPO}/{f}", doraise=True)
print("  OK sintaks semua file valid")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ninja"], check=True)
nproc = int(subprocess.run(["nproc"], capture_output=True, text=True).stdout.strip() or "2")
os.environ["MAX_JOBS"] = str(max(1, nproc))

# Cache wheel selective_scan ke Drive, ditandai arsitektur GPU + versi torch/python --
# ini satu-satunya langkah yang genuinely lambat (kompilasi CUDA). Clone+patch di atas
# sudah cepat & SELALU dijalankan fresh (supaya deform_ss2d.py terbaru & patch selalu
# terverifikasi) -- yang di-cache cuma hasil compile-nya, bukan langkah patch-nya.
cap = torch.cuda.get_device_capability(0)
SS_TAG = f"sm{cap[0]}{cap[1]}_torch{torch.__version__.split('+')[0]}_py{sys.version_info.major}{sys.version_info.minor}"
SS_WHEEL_DIR = f"{DRIVE_ROOT}/wheels/selective_scan_{SS_TAG}"
os.makedirs(SS_WHEEL_DIR, exist_ok=True)

cached_ss = glob.glob(f"{SS_WHEEL_DIR}/*.whl")
if cached_ss:
    print(f"Wheel selective_scan cache ditemukan ({SS_TAG}) -- install langsung, skip compile")
    subprocess.run([sys.executable, "-m", "pip", "install"] + cached_ss, check=True)
else:
    print(f"Belum ada cache utk {SS_TAG} -- compile sekali ini (~beberapa menit), lalu disimpan ke Drive")
    subprocess.run([sys.executable, "-m", "pip", "wheel", ".", "--no-deps", "-v", "-w", SS_WHEEL_DIR],
                   cwd=f"{REPO}/selective_scan", check=True)
    new_ss_wheels = glob.glob(f"{SS_WHEEL_DIR}/*.whl")
    subprocess.run([sys.executable, "-m", "pip", "install"] + new_ss_wheels, check=True)
    print(f"Tersimpan ke {SS_WHEEL_DIR} -- sesi berikutnya dgn GPU {SS_TAG.split('_')[0]} langsung cepat")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], cwd=REPO, check=True)

for k in [x for x in list(sys.modules) if x.startswith("ultralytics")]:
    del sys.modules[k]
import selective_scan_cuda_core
print("selective_scan_cuda_core berhasil di-import.")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "seaborn", "thop", "timm", "einops"], check=True)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q",
                "wandb", "ray", "tensorboard", "tensorboardX", "tensorboard-data-server",
                "clearml", "comet_ml", "dvclive", "mlflow", "neptune"])

yaml_path = f"{REPO}/ultralytics/cfg/datasets/dsf-yolo.yaml"
yaml_content = f"""path: {DSF_BASE}
train: images/train2021
val: images/val2021

nc: 8
names: [\'air-hole\', \'bite-edge\', \'broken-arc\', \'crack\', \'dense-gas-hole\', \'non-fusion\', \'slag-inclusion\', \'unfused-hole\']
"""
with open(yaml_path, "w") as f:
    f.write(yaml_content)

print("\nSEMUA PATCH + BUILD OK.")

Tripwire deform_ss2d.py OK

### Patch
  OK 1.1
  OK 1.2
  OK 1.3
  OK 1.4
  ~ 2.1: skip
  OK NP-1 ConvOffset.use_proj
  OK NP-2 DeformableScan.use_proj
  OK NP-3a DeformSS2D arg
  OK NP-3b teruskan use_proj
  OK NP-4 kelas DeformVSSBlockNP / DeformXSSBlockNP ditambahkan
  OK 3.1
  OK 3.2
  OK 3.3
  OK 4.1
  OK 4.2
  OK 5.1
  OK 2 YAML Ablasi B (backbone-only, neck-only) berhasil digenerate
  OK 3 YAML Ablasi B-lanjut (bb-deep, bb-late, bb-early) berhasil digenerate
  OK 1 YAML BB-Late-Alt (D-S-D, bersyarat) berhasil digenerate, indeks head diperbaiki
  OK 1 YAML P1-NoProj digenerate (4 DeformVSSBlockNP + 4 DeformXSSBlockNP)
  OK sintaks semua file valid
Wheel selective_scan cache ditemukan (sm80_torch2.11.0_py313) -- install langsung, skip compile
selective_scan_cuda_core berhasil di-import.

SEMUA PATCH + BUILD OK.


## 5. Pre-flight — Verifikasi Arsitektur

In [ ]:
os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)

from ultralytics import settings
settings.update({"tensorboard": False})

from ultralytics import YOLO
from ultralytics.nn.tasks import DetectionModel
from ultralytics.nn.modules.deform_ss2d import DeformSS2D
from ultralytics.nn.modules.mamba_yolo import SS2D

cfg_path = f"ultralytics/cfg/models/mamba-yolo/{CONFIG_YAML}"
m = DetectionModel(cfg_path, ch=3, nc=8, verbose=False)
n_par = sum(x.numel() for x in m.parameters())
n_def = sum(1 for x in m.modules() if isinstance(x, DeformSS2D))
n_plain = sum(1 for x in m.modules() if isinstance(x, SS2D) and not isinstance(x, DeformSS2D))

print(f"MODEL={MODEL}  SEED={SEED}  config={CONFIG_YAML}")
print(f"Params: {n_par/1e6:.3f}M (target {EXPECTED_PARAMS}M)  DeformSS2D: {n_def} (target {EXPECTED_NDEF})")
assert abs(n_par/1e6 - EXPECTED_PARAMS) < 0.01 and n_def == EXPECTED_NDEF, "Arsitektur tidak sesuai. STOP."

# ===== PRE-FLIGHT WAJIB KHUSUS P1-NoProj (PATCH-p1-noproj.md) =====
# "Kalau angkanya bukan 6.329.230, JANGAN mulai training."
if MODEL == "P1-NoProj":
    import torch.nn as nn
    from ultralytics.nn.modules.deform_ss2d import ConvOffset
    B2_PARAMS, TARGET = 5_986_912, 6_329_230

    print("\n=== PRE-FLIGHT P1-NoProj ===")
    print(f"  Params  : {n_par:,}")
    print(f"  Target  : {TARGET:,}")
    print(f"  Delta B2: {n_par - B2_PARAMS:+,} ({100*(n_par-B2_PARAMS)/B2_PARAMS:+.2f}%)")
    assert n_par == TARGET, f"MISMATCH! dapat {n_par:,}, harusnya {TARGET:,}"

    # (2) pastikan proj BENAR-BENAR hilang -- bukan cuma params kebetulan cocok
    n_co = sum(1 for mod in m.modules() if isinstance(mod, ConvOffset))
    n_id = sum(1 for mod in m.modules()
               if isinstance(mod, ConvOffset) and isinstance(mod.proj, nn.Identity))
    print(f"  ConvOffset tanpa proj: {n_id}/{n_co}")
    assert n_id == n_co == 10, f"harusnya 10/10 pakai Identity, dapat {n_id}/{n_co}"

    # (3) REGRESI: P1 lama TIDAK BOLEH berubah -- semua argumen baru default ke
    #     perilaku lama, jadi hasil 5-seed yg sudah terkumpul tetap reproducible.
    m_old = DetectionModel("ultralytics/cfg/models/mamba-yolo/DeMamba-YOLO-T.yaml",
                           ch=3, nc=8, verbose=False)
    n_old = sum(x.numel() for x in m_old.parameters())
    assert n_old == 5_930_869, f"P1 LAMA BERUBAH! {n_old:,} != 5,930,869 -- patch merusak reproducibility!"
    print(f"  Regresi P1 lama: {n_old:,} OK (tidak berubah)")
    del m_old

    # (4) ssm_ratio harus 2.0 (TIDAK dihandicap) -- inti varian ini.
    #     Dicek dgn membandingkan d_inner LANGSUNG ke B2, bukan menebak rasio.
    #     (d_inner bukan atribut; dibaca dari x_proj_weight.shape[-1])
    m_b2 = DetectionModel("ultralytics/cfg/models/mamba-yolo/Mamba-YOLO-T.yaml",
                          ch=3, nc=8, verbose=False)
    _np_inner = [x.x_proj_weight.shape[-1] for x in m.modules() if isinstance(x, DeformSS2D)]
    _b2_inner = [x.x_proj_weight.shape[-1] for x in m_b2.modules()
                 if isinstance(x, SS2D) and not isinstance(x, DeformSS2D)]
    print(f"  d_inner P1-NoProj: {_np_inner}")
    print(f"  d_inner B2       : {_b2_inner}")
    assert _np_inner == _b2_inner, (
        "d_inner BEDA dari B2 -> ssm_ratio bukan 2.0. Cek kelas *NP: pastikan TIDAK "
        "meng-override __init__ (itu yang dulu memaksa 1.4 di DeformVSSBlock lama).")
    del m_b2
    print("  PRE-FLIGHT P1-NoProj OK -- aman mulai training.")

del m
torch.cuda.empty_cache()
print("Pre-flight LOLOS.")

Overriding model.yaml nc=80 with nc=8
WARNING ⚠️ no model scale passed. Assuming scale='T'.


/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/content/Mamba-YOLO/ultralytics/nn/modules/common_utils_mbyolo.py:103: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd
/content/Mamba-YOLO/ultralytics/nn/modules/common_utils_mbyolo.py:130: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_bwd
/content/Mamba-YOLO/ultralytics/nn/modules/mamba_yolo.py:349: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in staticmethod() if you want to preserve the old behavior
  self.op = self.SSM_CLASS(
/c

MODEL=P1-NoProj  SEED=5  config=DeMamba-YOLO-T-noproj.yaml
Params: 6.329M (target 6.329M)  DeformSS2D: 10 (target 10)

=== PRE-FLIGHT P1-NoProj ===
  Params  : 6,329,230
  Target  : 6,329,230
  Delta B2: +342,318 (+5.72%)
  ConvOffset tanpa proj: 10/10
Overriding model.yaml nc=80 with nc=8
WARNING ⚠️ no model scale passed. Assuming scale='T'.
  Regresi P1 lama: 5,930,869 OK (tidak berubah)
Overriding model.yaml nc=80 with nc=8
WARNING ⚠️ no model scale passed. Assuming scale='T'.
  d_inner P1-NoProj: [64, 128, 256, 256, 256, 512, 256, 128, 256, 512]
  d_inner B2       : [64, 128, 256, 256, 256, 512, 256, 128, 256, 512]
  PRE-FLIGHT P1-NoProj OK -- aman mulai training.
Pre-flight LOLOS.


## 6. Cek Status Checkpoint — Baru / Belum Selesai / Sudah Selesai

Sel ini yang membuat notebook aman dijalankan ulang. Tidak ada training di sini, cuma
pengecekan status.

In [ ]:
from pathlib import Path

last_ckpt = Path(DRIVE_OUTPUT) / RUN_NAME / "weights" / "last.pt"

if not last_ckpt.exists():
    STATUS = "BARU"
    print(f"STATUS: {STATUS} -- belum ada checkpoint, training akan mulai dari nol.")
else:
    raw = torch.load(str(last_ckpt), map_location="cpu", weights_only=False)
    ep = raw.get("epoch", -1)
    has_opt = raw.get("optimizer") is not None
    if has_opt and ep >= 0 and ep < TARGET_EPOCHS - 1:
        STATUS = "BELUM_SELESAI"
        print(f"STATUS: {STATUS} -- checkpoint ada, epoch tersimpan={ep}, optimizer ada.")
        print(f"  -> Akan RESUME (genuine, LR schedule kontinu) dari epoch {ep+1}.")
    else:
        STATUS = "SELESAI"
        print(f"STATUS: {STATUS} -- checkpoint sudah final_eval (optimizer={has_opt}, epoch={ep}).")
        print("  -> Training TIDAK diulang. Lanjut ke Sel 8 untuk baca hasil.")

print(f"\nlast.pt: {last_ckpt}  (ada: {last_ckpt.exists()})")

STATUS: BARU -- belum ada checkpoint, training akan mulai dari nol.

last.pt: /content/drive/MyDrive/demamba-yolowd/output/p1-noproj_seed5_150ep_lr0.002/weights/last.pt  (ada: False)


## 7. Training — Mulai Baru atau Resume, Sesuai Status Sel 6

**`patience=0` WAJIB** — mematikan EarlyStopping. Proyek ini butuh jumlah epoch yang
**terkontrol ketat** (150, sama persis tiap run) untuk perbandingan B2 vs P1 yang adil.
Default Ultralytics `patience=100` membuat run berhenti otomatis begitu 100 epoch berlalu
tanpa perbaikan `fitness` — kalau dibiarkan, run yang berbeda bisa berhenti di jumlah epoch
berbeda-beda (satu model plateau lebih cepat dari yang lain), merusak prinsip
compute-matched/epoch-matched yang jadi dasar seluruh perbandingan proyek ini.

In [ ]:
if STATUS == "BARU":
    model = YOLO(cfg_path)
    model.train(
        data="ultralytics/cfg/datasets/dsf-yolo.yaml",
        epochs=TARGET_EPOCHS,
        batch=8, imgsz=640, device=0, amp=True, seed=SEED,
        patience=0,   # WAJIB: matikan EarlyStopping, perbandingan butuh epoch terkontrol ketat
        cache=True,   # percepat: dataset (~2700 gambar) muat di RAM, hindari re-decode JPEG tiap
                      # epoch. TIDAK mengubah data/hasil -- murni percepatan, aman utk perbandingan.
        optimizer="SGD",  # WAJIB: resep asli seed0 (mbyolo_train.py) pakai SGD, defaultnya.
                          # Tanpa ini, Ultralytics diam-diam pilih optimizer=auto (AdamW+LR
                          # auto) -- ditemukan sbg penyebab UTAMA divergensi P1 (bukan platform).
        cos_lr=USE_COS_LR,  # False (default eksperimen ini) -- kosinus TERBUKTI LEBIH BURUK,
                            # lihat catatan lengkap di Sel 3.
        lr0=LR0,  # WAJIB diturunkan dari default 0.01 -- lihat catatan lengkap di Sel 3.
                  # Ini yang menyerang akar masalah (nilai LR), bukan cuma bentuk kurvanya.
        # JANGAN PERNAH set time= -- itu override epochs= secara diam-diam (sama bahayanya dgn patience)
        project=DRIVE_OUTPUT, name=RUN_NAME,
    )
elif STATUS == "BELUM_SELESAI":
    model = YOLO(str(last_ckpt))
    # TANPA epochs=/patience=/optimizer= -- resume mengambil train_args tersimpan di checkpoint
    # (150, patience=0, optimizer=SGD -- asalkan run aslinya sudah pakai settingan ini di atas).
    model.train(resume=True)
else:
    model = YOLO(str(last_ckpt))
    print("Training sudah selesai sebelumnya -- sel ini di-skip, tidak training ulang.")

SAVE_DIR = model.trainer.save_dir if hasattr(model, "trainer") and model.trainer else Path(DRIVE_OUTPUT) / RUN_NAME
print(f"\nSAVE_DIR: {SAVE_DIR}")

# Verifikasi status akhir -- kalau sesi ini SENDIRI terputus, baris ini tidak akan tercetak,
# dan menjalankan ulang notebook (dari Sel 1) akan mendeteksi STATUS=BELUM_SELESAI otomatis.
final_check = torch.load(str(Path(SAVE_DIR) / "weights" / "last.pt"), map_location="cpu", weights_only=False)
if final_check.get("optimizer") is None:
    print(f"Training {RUN_NAME} SELESAI PENUH ({TARGET_EPOCHS} epoch).")
else:
    print(f"Training belum selesai (epoch {final_check.get('epoch')}) -- jalankan ulang notebook ini untuk lanjut.")

WARNING ⚠️ no model scale passed. Assuming scale='T'.
New https://pypi.org/project/ultralytics/8.4.155 available 😃 Update with 'pip install -U ultralytics'
Ultralytics YOLOv8.2.29 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: task=detect, mode=train, model=ultralytics/cfg/models/mamba-yolo/DeMamba-YOLO-T-noproj.yaml, data=ultralytics/cfg/datasets/dsf-yolo.yaml, epochs=150, time=None, patience=0, batch=8, imgsz=640, save=True, save_period=-1, cache=True, device=0, workers=8, project=/content/drive/MyDrive/demamba-yolowd/output, name=p1-noproj_seed5_150ep_lr0.002, exist_ok=False, pretrained=True, optimizer=SGD, verbose=True, seed=5, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, 

100%|██████████| 755k/755k [00:00<00:00, 29.8MB/s]

Overriding model.yaml nc=80 with nc=8
WARNING ⚠️ no model scale passed. Assuming scale='T'.

                   from  n    params  module                                       arguments                     
  0                  -1  1      5136  ultralytics.nn.modules.mamba_yolo.SimpleStem [3, 32, 3]                    
  1                  -1  1     34351  ultralytics.nn.modules.deform_ss2d.DeformVSSBlockNP[32, 32]                      
  2                  -1  1      8384  ultralytics.nn.modules.mamba_yolo.VisionClueMerge[32, 64]                      
  3                  -1  1    109083  ultralytics.nn.modules.deform_ss2d.DeformVSSBlockNP[64, 64]                      
  4                  -1  1     33152  ultralytics.nn.modules.mamba_yolo.VisionClueMerge[64, 128]                     
  5                  -1  3   1140285  ultralytics.nn.modules.deform_ss2d.DeformVSSBlockNP[128, 128]                    
  6                  -1  1    131840  ultralytics.nn.modules.mamba_yolo.VisionClueM


/content/Mamba-YOLO/ultralytics/nn/modules/mamba_yolo.py:349: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in staticmethod() if you want to preserve the old behavior
  self.op = self.SSM_CLASS(
/content/Mamba-YOLO/ultralytics/nn/modules/mamba_yolo.py:277: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in staticmethod() if you want to preserve the old behavior
  self.ss2d = nn.Sequential(*(self.SSM_CLASS(d_model=self.hidden_dim,


DeMamba-YOLO-T-noproj summary: 544 layers, 6329230 parameters, 6329214 gradients, 14.4 GFLOPs

Freezing layer 'model.21.dfl.conv.weight'
AMP: running Automatic Mixed Precision (AMP) checks with YOLOv8n...


...

  0%|          | 0/341 [00:00<?, ?it/s]/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
    150/150      7.47G      2.914      3.686      2.179          6        640: 100%|██████████| 341/341 [01:43<00:00,  3.31it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 43/43 [00:04<00:00,  9.67it/s]

                   all        682       1381     0.0114      0.162     0.0608     0.0205



150 epochs completed in 4.565 hours.
Optimizer stripped from /content/drive/MyDrive/demamba-yolowd/output/p1-noproj_seed5_150ep_lr0.002/weights/last.pt, 13.0MB
Optimizer stripped from /content/drive/MyDrive/demamba-yolowd/output/p1-noproj_seed5_150ep_lr0.002/weights/best.pt, 13.0MB

Validating /content/drive/MyDrive/demamba-yolowd/output/p1-noproj_seed5_150ep_lr0.002/weights/best.pt...
Ultralytics YOLOv8.2.29 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
DeMamba-YOLO-T-noproj summary: 528 layers, 6327886 parameters, 0 gradients, 14.3 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 43/43 [00:05<00:00,  7.67it/s]


                   all        682       1381      0.736      0.132       0.15     0.0431
              air-hole        406       1070      0.781     0.0234      0.079     0.0192
             bite-edge          8          8          1          0     0.0232     0.0032
            broken-arc         74         74          1          0       0.02    0.00783
                 crack         23         23          1          0          0          0
        dense-gas-hole         43         43          1          0          0          0
            non-fusion         40         41      0.775      0.587      0.727      0.183
        slag-inclusion         27         27          0          0          0          0
          unfused-hole         95         95      0.333      0.442      0.349      0.132
Speed: 0.1ms preprocess, 4.6ms inference, 0.0ms loss, 0.8ms postprocess per image
Results saved to /content/drive/MyDrive/demamba-yolowd/output/p1-noproj_seed5_150ep_lr0.002

SAVE_DIR: /content/drive

## 8. Baca Hasil (Hanya Kalau Sudah Selesai Penuh)

In [ ]:
import pandas as pd

df = pd.read_csv(Path(SAVE_DIR) / "results.csv")
df.columns = df.columns.str.strip()
last = df.iloc[-1]

print(f"=== {RUN_NAME} — {len(df)} baris tercatat (epoch terakhir: {int(df['epoch'].max())}) ===")
print(f"  mAP50    = {last['metrics/mAP50(B)']:.4f}")
print(f"  mAP50:95 = {last['metrics/mAP50-95(B)']:.4f}")
print(f"  P={last['metrics/precision(B)']:.4f}  R={last['metrics/recall(B)']:.4f}")

# Cek epoch TERAKHIR yang tercapai, BUKAN jumlah baris -- results.csv kadang bolong di
# tengah (mis. Drive I/O sempat gagal nulis 1-2 baris) walau training-nya sendiri utuh
# tanpa gangguan. len(df) < TARGET_EPOCHS bisa salah nunjuk "belum selesai" padahal
# training sudah genap sampai epoch target -- ditemukan dari kasus B2 seed4 (146 baris
# tapi epoch max=150, gap di epoch 26-27 & 87-88, loss mulus tanpa lompatan di sekitarnya).
epoch_tercapai = int(df['epoch'].max())
n_gap = TARGET_EPOCHS - len(df)
if epoch_tercapai < TARGET_EPOCHS:
    print(f"\n⚠️  Epoch terakhir cuma {epoch_tercapai}/{TARGET_EPOCHS} -- training BELUM selesai.")
    print("   Jalankan ulang notebook ini (Runtime > Run all) untuk lanjut.")
else:
    print(f"\nSELESAI PENUH -- epoch terakhir tercapai {epoch_tercapai}/{TARGET_EPOCHS}.")
    if n_gap > 0:
        missing = sorted(set(range(1, TARGET_EPOCHS+1)) - set(df['epoch'].astype(int)))
        print(f"  Catatan: {n_gap} baris log bolong di tengah (epoch {missing}) -- kemungkinan")
        print(f"  Drive I/O sempat gagal nulis, BUKAN training terputus. Cek loss di sekitar")
        print(f"  epoch itu kalau ragu (harus mulus, tanpa lompatan/reset).")
    print(f"Checkpoint: {Path(SAVE_DIR) / 'weights' / 'best.pt'}")
    print("Catat path ini untuk arsip & analisis agregat multi-seed nanti.")


=== p1-noproj_seed5_150ep_lr0.002 — 150 baris tercatat (epoch terakhir: 150) ===
  mAP50    = 0.0607
  mAP50:95 = 0.0205
  P=0.0114  R=0.1618

SELESAI PENUH -- epoch terakhir tercapai 150/150.
Checkpoint: /content/drive/MyDrive/demamba-yolowd/output/p1-noproj_seed5_150ep_lr0.002/weights/best.pt
Catat path ini untuk arsip & analisis agregat multi-seed nanti.


## 9. Breakdown Per-Kelas

Angka agregat (Sel 8) tidak cukup untuk Ablasi F (apakah keuntungan P1 terkonsentrasi di kelas
elongated/diagonal seperti crack, walau kalah di rata-rata karena air-hole yang dominan jumlahnya).
Sel ini WAJIB dijalankan tiap run selesai -- jangan cuma andalkan angka agregat Sel 8.

In [ ]:
import pandas as pd

model_final = YOLO(str(Path(SAVE_DIR) / "weights" / "best.pt"))
mf = model_final.val(data="ultralytics/cfg/datasets/dsf-yolo.yaml", split="val", verbose=False)

print(f"=== {RUN_NAME} -- per-kelas mAP50:95 ===")
print(f"Agregat: mAP50={mf.box.map50:.4f}  mAP50:95={mf.box.map:.4f}\n")
print(f"{'Kelas':16s} {'mAP50:95':>9s}")
for ci, cn in mf.names.items():
    print(f"{cn:16s} {mf.box.maps[ci]:>9.4f}")

# Simpan ke CSV (append, bukan timpa) -- MODEL & SEED diambil OTOMATIS dari variabel
# Sel 3, bukan diketik ulang manual (itu sumber human-error sebelumnya: lupa update
# nomor seed saat pakai ulang snippet utk kombinasi lain).
OUT_CSV = f"{DRIVE_ROOT}/analysis/per_class_results.csv"
Path(f"{DRIVE_ROOT}/analysis").mkdir(parents=True, exist_ok=True)

rows = [{
    "model": MODEL, "seed": SEED, "class": "ALL",
    "mAP50_95": mf.box.map, "mAP50": mf.box.map50,
    "precision": mf.box.mp, "recall": mf.box.mr,
    "ckpt_path": str(Path(SAVE_DIR) / "weights" / "best.pt"),
}]
for ci, cn in mf.names.items():
    rows.append({
        "model": MODEL, "seed": SEED, "class": cn,
        "mAP50_95": mf.box.maps[ci], "mAP50": None,
        "precision": None, "recall": None,
        "ckpt_path": str(Path(SAVE_DIR) / "weights" / "best.pt"),
    })
df_new = pd.DataFrame(rows)

if Path(OUT_CSV).exists():
    df_old = pd.read_csv(OUT_CSV)
    df_out = pd.concat([df_old, df_new], ignore_index=True)
    df_out = df_out.drop_duplicates(subset=["model", "seed", "class"], keep="last")
else:
    df_out = df_new
df_out.to_csv(OUT_CSV, index=False)
print(f"\nTersimpan ke: {OUT_CSV}  ({len(df_out)} baris total, {MODEL} seed{SEED})")


Ultralytics YOLOv8.2.29 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
DeMamba-YOLO-T-noproj summary: 528 layers, 6327886 parameters, 0 gradients, 14.3 GFLOPs


val: Scanning /content/data/steel-tube-dataset-all/yolo/labels/val2021.cache... 682 images, 2 backgrounds, 0 corrupt: 100%|██████████| 682/682 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 43/43 [00:06<00:00,  6.31it/s]


                   all        682       1381      0.744      0.132      0.155     0.0434
Speed: 0.1ms preprocess, 6.1ms inference, 0.0ms loss, 1.1ms postprocess per image
Results saved to /content/Mamba-YOLO/runs/detect/val
=== p1-noproj_seed5_150ep_lr0.002 -- per-kelas mAP50:95 ===
Agregat: mAP50=0.1548  mAP50:95=0.0434

Kelas             mAP50:95
air-hole            0.0195
bite-edge           0.0058
broken-arc          0.0079
crack               0.0000
dense-gas-hole      0.0000
non-fusion          0.1830
slag-inclusion      0.0000
unfused-hole        0.1312

Tersimpan ke: /content/drive/MyDrive/demamba-yolowd/analysis/per_class_results.csv  (360 baris total, P1-NoProj seed5)


## Checklist

- [ ] Sel 1-2: Drive mount, dataset Kaggle, GPU terdeteksi
- [ ] Sel 3: konfigurasi MODEL/SEED benar
- [ ] Sel 4-5: patch + pre-flight LOLOS
- [ ] Sel 6: status checkpoint terbaca (BARU / BELUM_SELESAI / SELESAI)
- [ ] Sel 7: training jalan sesuai status (baru/resume/skip)
- [ ] Sel 8: **150/150 epoch** — kalau belum, ulangi Sel 1-8 (Run all) sampai selesai
- [ ] Sel 9: breakdown per-kelas tercatat (WAJIB, jangan skip -- dibutuhkan utk Ablasi F nanti)
- [ ] Centang baris di tabel progres (Sel 0)

**Catatan metodologi untuk paper:** seed 1-4 (notebook ini) = 150 epoch kontinu, LR schedule
tunggal. Seed 0 (Kaggle, sudah ada) = 100+50 dengan LR-restart. Perbedaan ini WAJIB
disebutkan eksplisit di bagian metodologi — pertimbangkan apakah seed 0 perlu diulang bersih
untuk konsistensi penuh sebelum menulis draft final.

## 10. Matikan Runtime (Wajib Dijalankan Terakhir)

In [ ]:
# Matikan runtime Colab otomatis setelah semua sel di atas selesai --
# supaya TIDAK terus memakan kuota Colab Pro kalau kamu tidak sempat
# matikan manual. Ini API resmi Google, benar-benar melepas VM-nya.
print("Semua proses selesai. Runtime akan di-disconnect otomatis...")
import time
time.sleep(10)  # jeda singkat, jaga-jaga ada file yang masih proses ditulis ke Drive

try:
    from google.colab import runtime
    runtime.unassign()
except Exception as e:
    print(f"Gagal disconnect otomatis ({e})")
    print("Matikan manual lewat: Runtime > Disconnect and delete runtime")
